# Final Usage E8 + translation refit

Train **one fresh E8 model on all 32,772 eligible teacher development images**.
Use translation and class weights from the five saved E8 recipes. Save epoch **30**.

Open this notebook in **VS Code** and select your **Colab GPU kernel**.
Fetch the branch code from GitHub. Reuse the existing Drive ZIP.
No old checkpoint is loaded. No holdout or test score is produced here.

## 1. Mount Drive and choose paths

In [ ]:
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

from google.colab import drive

REPO_URL = "https://github.com/TrnLin/MLA2.git"
BRANCH = "task-3-gender-usage-classification"
REPO_DIR = Path("/content/MLA2")
DRIVE_PROJECT = Path("/content/drive/MyDrive/MLA2")
DATA_ZIP = DRIVE_PROJECT / "data/task3-data.zip"
DRIVE_TASK_DIR = DRIVE_PROJECT / "task3"
DRIVE_REGISTRY = DRIVE_TASK_DIR / "results/runs.csv"
LOCAL_REGISTRY = REPO_DIR / "results/runs.csv"
drive.mount("/content/drive", force_remount=False)

## 2. Fetch the published code

Use a fresh kernel so imports match this checkout.
Later code changes need a GitHub push; they do not need another data ZIP.

In [ ]:
def run_checked(command):
    return subprocess.run([str(x) for x in command], check=True)


if (REPO_DIR / ".git").is_dir():
    remote = subprocess.check_output(
        ["git", "-C", str(REPO_DIR), "remote", "get-url", "origin"], text=True
    ).strip()
    if remote != REPO_URL:
        raise RuntimeError("The local checkout belongs to another repository")
    run_checked(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH])
    run_checked(["git", "-C", REPO_DIR, "switch", BRANCH])
    run_checked(["git", "-C", REPO_DIR, "merge", "--ff-only", f"origin/{BRANCH}"])
else:
    run_checked(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR])
os.chdir(REPO_DIR)
os.environ["FASHION_PROJECT_ROOT"] = str(REPO_DIR)
sys.path.insert(0, str(REPO_DIR / "src"))
print("Code commit:")
run_checked(["git", "rev-parse", "HEAD"])

## 3. Reuse the existing teacher ZIP

Read eligible development paths from the canonical split.
Extract missing images only. Keep GitHub's split, class map and small E8 recipe snapshot.

In [ ]:
import pandas as pd

from fashion.data import get_samples, load_splits

splits = load_splits(REPO_DIR / "data/processed/splits.csv")
paths = get_samples(splits, partition="development", target="usage").path.tolist()
missing = [source_path for source_path in paths if not (REPO_DIR / source_path).is_file()]
if missing:
    local_zip = Path("/content/task3-usage-e8-refit-data.zip")
    shutil.copyfile(DATA_ZIP, local_zip)
    with zipfile.ZipFile(local_zip) as archive:
        for source_path in missing:
            if not source_path.startswith("data/raw/teacher/train/images_train/"):
                raise ValueError(f"Unexpected development image path: {source_path}")
            target = (REPO_DIR / source_path).resolve()
            if not target.is_relative_to(REPO_DIR.resolve()):
                raise ValueError("Image path leaves the repository")
            target.parent.mkdir(parents=True, exist_ok=True)
            partial = target.with_suffix(target.suffix + ".partial")
            with archive.open(source_path) as source, partial.open("wb") as output:
                shutil.copyfileobj(source, output)
            partial.replace(target)
print(f"Development images ready: {len(paths):,}")

## 4. Check E8 before training

SmallCNN: **391,209 parameters**, global average pooling, no dropout.
Training translation: a **50% chance** of shifting each axis by an integer from **-2 to 2 pixels**.
Keep the original white-fill image transform, then RGB input at height 80, width 60.

Use effective-number weighted cross-entropy: **beta 0.999**, **cap 5.0**.
Weights come from the full development class counts. Normalization uses only
full-development content pixels. Neither fit uses protected rows.

AdamW: rate **0.001**, weight decay **0.0001**, batch **128**, seed **2753**.
Train **30 epochs**, cosine T_max **30**, minimum rate **0.00001**.
No MixUp, SAM, expanded data, validation set or early stopping.

In [ ]:
import torch
from fashion.train.task3_usage_e8_refit import EXPERIMENT, prepare_refit

config, contract, training = prepare_refit(root=REPO_DIR)
print("Rows:", len(training), "Classes:", contract["class_names"])
print("Translation:", contract["training_augmentation"])
print("Weight recipe:", contract["class_weight_contract"])
print("PyTorch:", torch.__version__)
print("Output:", DRIVE_TASK_DIR / "experiments" / EXPERIMENT / "usage")

## 5. Train and save one fixed model

This cell starts the real fit. Each epoch visits every admitted row once.
Save epoch 30. Keep training loss separate from evaluation scores.

The Drive registry is the main log. The checkout log is a mirror during training.
A completed rerun checks the model files and main log, then returns without writing or fitting.
An interrupted or failed run stops for inspection. It does not silently restart.

In [ ]:
from fashion.train.task3_usage_e8_refit import run_usage_e8_refit

result = run_usage_e8_refit(
    root=REPO_DIR,
    output_root=DRIVE_TASK_DIR,
    registry_path=DRIVE_REGISTRY,
    registry_mirrors=(LOCAL_REGISTRY,),
)
print("Status:", result["status"], "Reused:", result["reused"])
print("Manifest:", result["manifest_path"])

## 6. Check the saved files

In [ ]:
manifest_dir = Path(result["manifest_path"]).parent
history = pd.read_csv(manifest_dir / "history.csv")
assert history.epoch.tolist() == list(range(1, 31))
assert history.training_rows.eq(32772).all()
assert history.selected_checkpoint.tolist() == [False] * 29 + [True]
assert result["training_completed"] and not result["evaluation_completed"]
display(history.tail())
for name in ("final_epoch.pt", "normalization.json", "class_weights.json", "model_manifest.json"):
    print(name, manifest_dir / name)

## 7. Hand off for later judgement

Verify the manifest's file hashes. Rebuild `Task3BaselineCNN` from the saved `base_config`
and strictly load this checkpoint's weights. Use its saved normalization and nine-class map.
Predict with one model in evaluation mode, without random translation.
Take softmax then argmax. The training class weights are not prediction multipliers.

Write any later evaluation to a new evidence folder linked to this checkpoint hash.
Report the new model's own scores, class failures and measured cost.
Old E8 validation macro-F1 **0.419393** belongs to the five fold models, not this refit.

Holdout and recovered test labels were already viewed. Later scoring is **not newly blind**.
Do not use E1's recent holdout results to change this fixed E8 recipe.
This notebook does not evaluate the model, replace E1 or write a submission.